# Evaluasi Model Klasifikasi SIBI/BISINDO

Buku catatan (notebook) ini berisi skrip evaluasi model *machine learning* untuk mendeteksi abjad. Tugas dari bagian evaluasi ini adalah:
1. Menghitung Accuracy, Precision, Recall, F1-Score
2. Membuat visualisasi Confusion Matrix
3. Menganalisis huruf yang paling sering salah terdeteksi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf

# Daftar huruf A-Z untuk label
kelas_huruf = [chr(i) for i in range(ord('A'), ord('Z')+1)]


In [ ]:
# --- BAGIAN INI UNTUK DATA ASLI NANTI ---
# Jika dataset uji (test_data.csv) dan model (model.h5) sudah diberikan oleh teman sekelompok (Aldyth/Fadhil), 
# hilangkan komentar (uncomment) pada baris di bawah ini dan sesuaikan jalurnya:

# model = tf.keras.models.load_model('../models/model_bisindo.h5')
# test_df = pd.read_csv('../data/processed/test_landmarks.csv')
# X_test = test_df.drop('label', axis=1).values
# y_test_asli = test_df['label'].values
# y_pred_prob = model.predict(X_test)
# y_pred = np.argmax(y_pred_prob, axis=1)

# --- BAGIAN DUMMY DATA (Hapus jika data asli sudah siap) ---
# Karena model dan data asli belum ada, kita buat data "pura-pura" 
# agar kode evaluasi dan grafiknya bisa dites dan dilihat dosen!

np.random.seed(42)
jumlah_sampel = 500

# Buat array berisi index 0 sampai 25 (mewakili A-Z) secara acak
y_test_asli = np.random.randint(0, 26, jumlah_sampel)

# Buat prediksi dari model (90% dibuat sama dengan asli agar seolah akurasinya 90%)
y_pred = y_test_asli.copy()
jumlah_salah = int(0.10 * jumlah_sampel) # 10% diprediksi salah
index_salah = np.random.choice(jumlah_sampel, jumlah_salah, replace=False)

for i in index_salah:
    # Buat sengaja salah tebak, misalnya M (12) jadi N (13) atau sebaliknya
    if y_test_asli[i] == 12: y_pred[i] = 13
    elif y_test_asli[i] == 13: y_pred[i] = 12
    elif y_test_asli[i] == 4: y_pred[i] = 0 # E (4) disangka A (0)
    else:
        y_pred[i] = np.random.randint(0, 26) # salah acak
        
print("Data berhasil disiapkan!")


In [ ]:
# 1. Menghitung Akurasi
akurasi = accuracy_score(y_test_asli, y_pred)
print(f"Akurasi Keseluruhan (Overall Accuracy): {akurasi * 100:.2f}%\n")

# 2. Menghitung Precision, Recall, F1-Score per Kelas
# Gunakan classification_report dari scikit-learn
laporan = classification_report(y_test_asli, y_pred, target_names=kelas_huruf)
print("--- Laporan Klasifikasi per Huruf ---")
print(laporan)


In [ ]:
# 3. Membuat Confusion Matrix 26x26
cm = confusion_matrix(y_test_asli, y_pred)

plt.figure(figsize=(15, 12))
# Gunakan heatmap dari seaborn agar berwarna dan mudah dibaca
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=kelas_huruf, 
            yticklabels=kelas_huruf)

plt.title('Confusion Matrix Deteksi BISINDO', fontsize=16)
plt.ylabel('Label Sebenarnya (True Label)', fontsize=12)
plt.xlabel('Label Prediksi (Predicted Label)', fontsize=12)
plt.show()


In [ ]:
# 4. Menganalisis Huruf yang Paling Sering Tertukar
# Kita akan mencari cell di confusion matrix yang nilainya tinggi tapi bukan di diagonal utama (bukan tebakan benar)

print("--- Analisis Huruf yang Sering Tertukar ---")
kesalahan = []

# Ulangi untuk tiap baris dan kolom
for i in range(26):
    for j in range(26):
        if i != j and cm[i, j] > 0: # jika prediksi salah dan ada jumlahnya
            kesalahan.append({
                'asli': kelas_huruf[i],
                'ditebak_sebagai': kelas_huruf[j],
                'jumlah': cm[i, j]
            })

# Urutkan dari jumlah kesalahan yang paling banyak
kesalahan = sorted(kesalahan, key=lambda x: x['jumlah'], reverse=True)

# Tampilkan 5 kesalahan terbesar
print("Top 5 Huruf Paling Sering Tertukar:")
for k, err in enumerate(kesalahan[:5]):
    print(f"{k+1}. Huruf '{err['asli']}' malah terdeteksi sebagai '{err['ditebak_sebagai']}' sebanyak {err['jumlah']} kali.")
